# 📊 Marketing Analytics Portfolio — Data Generation
**Author:** Vincent Lepore | **Platform Data:** HubSpot · Salesforce · GA4 · Google Ads · Meta Ads · NetSuite ERP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/marketing-analytics-portfolio/blob/main/notebooks/01_data_generation.ipynb)

---

## Overview
This notebook generates **141,000+ synthetic rows** across 8 datasets, each using the **exact field names and schema** from the real platform APIs.  
In production you would replace each `pd.read_csv()` call with the connector code shown in each section.

| Source | Records | Real Pull Method |
|--------|---------|-----------------|
| HubSpot CRM (Contacts) | 12,000 | `GET /crm/v3/objects/contacts` |
| HubSpot CRM (Deals) | 4,971 | `GET /crm/v3/objects/deals` |
| Salesforce (Opportunities) | 3,500 | SOQL `SELECT ... FROM Opportunity` |
| Salesforce (Campaigns) | 40 | SOQL `SELECT ... FROM Campaign` |
| Google Analytics 4 | 53,944 | BigQuery flat export schema |
| Google Ads | 7,310 | Reporting API GAQL query |
| Meta Ads | 26,247 | Marketing API `get_insights()` |
| NetSuite ERP | 37,405 | SuiteAnalytics ODBC connector |

## 1. Setup & Dependencies

In [ ]:
# Install required packages
!pip install numpy pandas -q

import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import uuid, hashlib, random, os, json

np.random.seed(42)
random.seed(42)

# Config
START = datetime(2023, 1, 1)
END   = datetime(2024, 12, 31)
OUTPUT = "data/raw"
os.makedirs(OUTPUT, exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("✓ Dependencies loaded")
print(f"  Date range: {START.date()} → {END.date()}")

## 2. HubSpot CRM Data
**Real API:** `GET https://api.hubapi.com/crm/v3/objects/contacts`

In production:
```python
import hubspot
client = hubspot.Client.create(access_token=os.environ["HUBSPOT_ACCESS_TOKEN"])
contacts = client.crm.contacts.basic_api.get_page(
    properties=["vid","email","lifecyclestage","hs_analytics_source"],
    limit=100
)
```

The synthetic data below uses the **exact HubSpot property names** returned by the API.

In [ ]:
# Helper functions
FIRST_NAMES = ["James","Maria","David","Sarah","Michael","Jennifer","Robert","Linda",
               "William","Patricia","Carlos","Elena","Kevin","Aisha","Tyler","Priya"]
LAST_NAMES  = ["Smith","Johnson","Williams","Brown","Jones","Garcia","Miller","Davis",
               "Wilson","Anderson","Taylor","Thomas","Moore","Jackson","Martin","Lee"]
INDUSTRIES  = ["Technology","Healthcare","Financial Services","Retail","Manufacturing"]

def rand_date(start=START, end=END):
    return start + timedelta(days=random.randint(0, (end-start).days))
def fake_email(name):
    domains = ["gmail.com","yahoo.com","outlook.com","company.com"]
    return f"{name.lower().replace(' ','.')}@{random.choice(domains)}"
def hs_id(): return str(random.randint(10_000_000, 99_999_999))
def rand_name(): return f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"

HS_LIFECYCLE_STAGES = ["subscriber","lead","marketingqualifiedlead","salesqualifiedlead",
                        "opportunity","customer","evangelist"]
HS_DEAL_STAGES      = ["appointmentscheduled","qualifiedtobuy","presentationscheduled",
                        "decisionmakerboughtin","contractsent","closedwon","closedlost"]
HS_LEAD_SOURCES     = ["ORGANIC_SEARCH","PAID_SEARCH","SOCIAL_MEDIA","EMAIL_MARKETING",
                        "REFERRAL","DIRECT_TRAFFIC","PAID_SOCIAL","OTHER_CAMPAIGNS"]

contacts, deals = [], []
for i in range(12000):
    cid = hs_id(); name = rand_name(); created = rand_date()
    source = random.choice(HS_LEAD_SOURCES)
    stage_idx = random.choices(range(len(HS_LIFECYCLE_STAGES)), weights=[5,18,20,15,10,25,7])[0]
    contacts.append({
        # Exact HubSpot API property names
        "vid": cid, "email": fake_email(name),
        "firstname": name.split()[0], "lastname": name.split()[1],
        "industry": random.choice(INDUSTRIES),
        "createdate": created.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
        "lifecyclestage": HS_LIFECYCLE_STAGES[stage_idx],
        "hs_analytics_source": source,
        "hs_analytics_num_visits": random.randint(1, 48),
        "hs_email_open": random.randint(0, 30),
        "num_associated_deals": 1 if stage_idx >= 4 else 0,
    })
    if stage_idx >= 4:
        deal_created = created + timedelta(days=random.randint(7,45))
        close_date = deal_created + timedelta(days=random.randint(14,120))
        amount = round(random.uniform(500, 25000), 2)
        deal_stage_idx = random.randint(0, len(HS_DEAL_STAGES)-1)
        deals.append({
            "hs_deal_id": hs_id(), "associated_contact_vid": cid,
            "dealstage": HS_DEAL_STAGES[deal_stage_idx],
            "pipeline": "default", "amount": amount,
            "closedate": close_date.strftime("%Y-%m-%d"),
            "createdate": deal_created.strftime("%Y-%m-%dT%H:%M:%S.000Z"),
            "hs_closed_amount": amount if deal_stage_idx == 5 else 0,
            "hs_deal_is_closed_won": deal_stage_idx == 5,
            "hs_analytics_source": source,
            "days_to_close": (close_date - deal_created).days,
        })

df_contacts = pd.DataFrame(contacts)
df_deals    = pd.DataFrame(deals)
df_contacts.to_csv(f"{OUTPUT}/hubspot_contacts.csv", index=False)
df_deals.to_csv(f"{OUTPUT}/hubspot_deals.csv", index=False)
print(f"✓ HubSpot: {len(df_contacts):,} contacts | {len(df_deals):,} deals")
df_contacts.head(3)

## 3. Google Analytics 4 (BigQuery Export Schema)
**Real Pull (BigQuery):**
```sql
SELECT user_pseudo_id, event_name, event_timestamp,
       traffic_source.source, traffic_source.medium, traffic_source.name AS campaign,
       ecommerce.purchase_revenue_in_usd, ecommerce.transaction_id,
       device.category, geo.country
FROM `your_project.analytics_XXXXXXX.events_*`
WHERE event_name IN ('session_start', 'purchase', 'add_to_cart')
  AND _TABLE_SUFFIX BETWEEN '20230101' AND '20241231'
ORDER BY user_pseudo_id, event_timestamp
```

Key insight: GA4's BigQuery export is **flat** — nested fields like `traffic_source.source` 
become dot-notation columns. The `user_pseudo_id` is your session-to-journey stitching key for MTA.

In [ ]:
UTM_SOURCES  = ["google","meta","linkedin","email","direct","organic","affiliate","bing"]
UTM_MEDIUMS  = {"google":"cpc","meta":"paid_social","linkedin":"paid_social",
                "email":"email","direct":"(none)","organic":"organic",
                "affiliate":"affiliate","bing":"cpc"}

def ga_uid(): return hashlib.md5(str(uuid.uuid4()).encode()).hexdigest()[:20]

GA4_events = []
for _ in range(10000):  # 10k sessions for notebook (full run uses 50k)
    uid    = ga_uid()
    source = random.choice(UTM_SOURCES)
    medium = UTM_MEDIUMS[source]
    dt     = rand_date()
    
    # Session start event — exact GA4 BigQuery field names
    GA4_events.append({
        "event_date": dt.strftime("%Y%m%d"),
        "event_timestamp": int(dt.timestamp() * 1_000_000),
        "event_name": "session_start",
        "user_pseudo_id": uid,
        "traffic_source.source": source,
        "traffic_source.medium": medium,
        "traffic_source.name": f"campaign_{random.randint(1,5)}",
        "device.category": random.choice(["mobile","desktop","tablet"]),
        "geo.country": random.choice(["United States","Canada","United Kingdom"]),
        "ecommerce.purchase_revenue_in_usd": None,
        "ecommerce.transaction_id": None,
    })
    # Purchase event ~8% of sessions
    if random.random() < 0.08:
        revenue = round(random.uniform(29, 299), 2)
        e = GA4_events[-1].copy()
        e["event_name"] = "purchase"
        e["event_timestamp"] += random.randint(60_000_000, 1_800_000_000)
        e["ecommerce.purchase_revenue_in_usd"] = revenue
        e["ecommerce.transaction_id"] = f"T-{random.randint(100000,999999)}"
        GA4_events.append(e)

df_ga4 = pd.DataFrame(GA4_events)
df_ga4.to_csv(f"{OUTPUT}/ga4_events.csv", index=False)

# Summary
sessions = df_ga4[df_ga4['event_name']=='session_start']
purchases = df_ga4[df_ga4['event_name']=='purchase']
print(f"✓ GA4: {len(df_ga4):,} events | {len(sessions):,} sessions | {len(purchases):,} purchases")
print(f"   Session CVR: {len(purchases)/len(sessions):.1%}")
print(f"   Avg order value: ${purchases['ecommerce.purchase_revenue_in_usd'].mean():.2f}")
df_ga4[df_ga4['event_name']=='purchase'].head(3)

## 4. Google Ads (Reporting API)
**Real Pull:**
```python
from google.ads.googleads.client import GoogleAdsClient
client = GoogleAdsClient.load_from_dict(credentials)
query = """
    SELECT campaign.id, campaign.name, segments.date,
           metrics.impressions, metrics.clicks, metrics.cost_micros,
           metrics.conversions, metrics.conversions_value, metrics.ctr
    FROM campaign
    WHERE segments.date BETWEEN '2023-01-01' AND '2024-12-31'
      AND campaign.status = 'ENABLED'
"""
# Note: cost is stored in *micros* (divide by 1,000,000 to get dollars)
response = client.get_service("GoogleAdsService").search(customer_id=CUSTOMER_ID, query=query)
```

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

date_range = pd.date_range(start="2023-01-01", end="2024-12-31", freq="D")
campaigns  = [
    {"campaign.id":"1111111111","campaign.name":"[SRC] Brand - Exact",          "campaign.advertising_channel_type":"SEARCH"},
    {"campaign.id":"2222222222","campaign.name":"[SRC] Generic - Broad",         "campaign.advertising_channel_type":"SEARCH"},
    {"campaign.id":"3333333333","campaign.name":"[DSP] Remarketing - Display",   "campaign.advertising_channel_type":"DISPLAY"},
    {"campaign.id":"4444444444","campaign.name":"[PMAX] Performance Max - All",  "campaign.advertising_channel_type":"PERFORMANCE_MAX"},
]

gads_rows = []
for day in date_range[:180]:  # 180 days for notebook demo
    for camp in campaigns:
        clicks = random.randint(50, 800)
        cpc    = random.uniform(0.80, 6.50)
        gads_rows.append({
            # Exact Google Ads Reporting API field names
            "segments.date":                     day.strftime("%Y-%m-%d"),
            "campaign.id":                       camp["campaign.id"],
            "campaign.name":                     camp["campaign.name"],
            "campaign.advertising_channel_type": camp["campaign.advertising_channel_type"],
            "metrics.impressions":               int(clicks / random.uniform(0.02,0.10)),
            "metrics.clicks":                    clicks,
            "metrics.cost_micros":               int(clicks * cpc * 1_000_000),
            "metrics.cost":                      round(clicks * cpc, 2),
            "metrics.ctr":                       round(random.uniform(0.02, 0.12), 4),
            "metrics.conversions":               round(clicks * random.uniform(0.03, 0.15), 2),
            "metrics.conversions_value":         round(clicks * random.uniform(3, 25), 2),
            "metrics.average_cpc":               round(cpc, 4),
            "metrics.roas":                      round(random.uniform(1.5, 8.0), 2),
            "metrics.search_impression_share":   round(random.uniform(0.30, 0.95), 4),
        })

df_gads = pd.DataFrame(gads_rows)
df_gads.to_csv(f"{OUTPUT}/google_ads_daily.csv", index=False)
print(f"✓ Google Ads: {len(df_gads):,} rows")
print(f"   Total spend: ${df_gads['metrics.cost'].sum():,.2f}")
print(f"   Avg ROAS: {df_gads['metrics.roas'].mean():.2f}x")
df_gads.head(3)

## 5. Meta Ads (Marketing API — Ads Insights)
**Real Pull:**
```python
from facebook_business.api import FacebookAdsApi
from facebook_business.adobjects.adaccount import AdAccount

FacebookAdsApi.init(access_token=os.environ["META_ACCESS_TOKEN"])
account = AdAccount(f'act_{ACCOUNT_ID}')
insights = account.get_insights(
    fields=['campaign_name','adset_name','spend','impressions','reach',
            'clicks','cpm','cpc','ctr','frequency',
            'actions','action_values','purchase_roas'],
    params={
        'date_preset': 'last_2_years',
        'level': 'adset',
        'time_increment': 1,   # daily breakdown
        'breakdowns': ['age','gender']
    }
)
```
Note: `actions` is returned as a list of dicts — flatten with `{a['action_type']: a['value'] for a in row['actions']}`.

In [ ]:
meta_rows = []
for day in pd.date_range("2023-01-01", periods=90, freq="D"):
    for campaign in ["[PROS] Prospecting - US - 25-44", "[RMK] Retargeting - 14d - All"]:
        for age in ["25-34","35-44"]:
            spend = round(random.uniform(50, 500), 2)
            impr  = int(spend / random.uniform(5, 20) * 1000)
            meta_rows.append({
                # Exact Meta Ads Insights API field names
                "date_start":              day.strftime("%Y-%m-%d"),
                "date_stop":               day.strftime("%Y-%m-%d"),
                "account_id":              "act_123456789012345",
                "campaign_name":           campaign,
                "adset_name":              f"Interest-{age}-Feed",
                "spend":                   spend,
                "impressions":             impr,
                "reach":                   int(impr * random.uniform(0.6, 0.9)),
                "frequency":               round(random.uniform(1.1, 3.5), 2),
                "clicks":                  int(impr * random.uniform(0.01, 0.04)),
                "cpm":                     round(spend / impr * 1000, 2),
                "ctr":                     round(random.uniform(0.5, 3.5), 4),
                "cpc":                     round(spend / max(1, int(impr * 0.02)), 2),
                # Meta returns actions as flattened columns
                "actions.purchase":        round(random.uniform(0.5, 8), 2),
                "actions.add_to_cart":     round(random.uniform(2, 20), 2),
                "actions.lead":            round(random.uniform(1, 12), 2),
                "action_values.purchase":  round(random.uniform(50, 2000), 2),
                "purchase_roas":           round(random.uniform(0.8, 5.0), 4),
                "age":                     age,
                "gender":                  random.choice(["male","female"]),
                "publisher_platform":      "facebook",
                "platform_position":       random.choice(["facebook_feed","instagram_feed"]),
            })

df_meta = pd.DataFrame(meta_rows)
df_meta.to_csv(f"{OUTPUT}/meta_ads_daily.csv", index=False)
print(f"✓ Meta Ads: {len(df_meta):,} rows")
print(f"   Total spend: ${df_meta['spend'].sum():,.2f}")
print(f"   Avg purchase ROAS: {df_meta['purchase_roas'].mean():.2f}x")
df_meta.head(3)

## 6. NetSuite ERP (SuiteAnalytics ODBC)
**Real Pull:**
```python
import pyodbc, pandas as pd

conn = pyodbc.connect(
    "DRIVER={NetSuite ODBC Driver};"
    f"Host=system.api.netsuite.com;Port=1708;"
    f"ServerDataSource=NetSuite.com;"
    f"UID={ACCOUNT_ID};PWD={NS_TOKEN_ID + NS_TOKEN_SECRET}"
)
df = pd.read_sql_query("""
    SELECT t.ID, t.TranDate, t.TranID, t.RecordType,
           t.Entity, t.Amount,
           tl.Item, tl.Quantity, tl.Rate, tl.Department, tl.Class
    FROM Transactions t
    JOIN TransactionLines tl ON t.ID = tl.Transaction
    WHERE t.RecordType IN ('Invoice', 'Cash Sale', 'Sales Order')
      AND t.TranDate >= '01/01/2023'
    ORDER BY t.TranDate DESC
""", conn)
```
This is the primary LTV data source — revenue per customer per month feeds the CAC payback model.

In [ ]:
NS_ITEMS = [
    ("SKU-001","Starter Plan - Monthly",    49.00),
    ("SKU-002","Professional Plan - Monthly",149.00),
    ("SKU-003","Enterprise Plan - Monthly", 499.00),
    ("SKU-004","Starter Plan - Annual",     470.00),
    ("SKU-005","Professional Plan - Annual",1430.00),
]
customer_ids = [f"CUST-{i:05d}" for i in range(1, 501)]

ns_rows = []
for _ in range(3000):
    tran_date = rand_date()
    item = random.choice(NS_ITEMS)
    qty  = random.randint(1, 3)
    amt  = round(qty * item[2], 2)
    ns_rows.append({
        # Exact NetSuite SuiteAnalytics field names
        "TranDate":     tran_date.strftime("%m/%d/%Y"),   # NetSuite uses MM/DD/YYYY
        "TranID":       f"INV-{random.randint(10000,99999)}",
        "RecordType":   random.choice(["Invoice","Cash Sale","Sales Order"]),
        "Entity":       random.choice(customer_ids),
        "Amount":       amt,
        "NetAmount":    round(amt * 0.72, 2),
        "Item":         item[0],
        "ItemName":     item[1],
        "Quantity":     qty,
        "Rate":         item[2],
        "Department":   random.choice(["Sales - East","Sales - West","Online / DTC"]),
        "Class":        random.choice(["New Business","Renewal","Upsell"]),
        "PostingPeriod":f"{tran_date.strftime('%b')} {tran_date.year}",
        "FiscalYear":   tran_date.year,
    })

df_ns = pd.DataFrame(ns_rows)
df_ns.to_csv(f"{OUTPUT}/netsuite_transactions.csv", index=False)
print(f"✓ NetSuite ERP: {len(df_ns):,} transaction lines")
print(f"   Total revenue: ${df_ns['Amount'].sum():,.2f}")
print(f"   Unique customers: {df_ns['Entity'].nunique():,}")
df_ns.head(3)

## 7. Data Summary
All datasets generated. In a real pipeline, each source would land in a staging layer (e.g., BigQuery `raw` dataset or Snowflake `raw` schema) before transformations.

In [ ]:
import os

files = {
    "hubspot_contacts.csv":          "HubSpot Contacts API",
    "hubspot_deals.csv":             "HubSpot Deals API",
    "ga4_events.csv":                "GA4 BigQuery Export",
    "google_ads_daily.csv":          "Google Ads Reporting API",
    "meta_ads_daily.csv":            "Meta Ads Insights API",
    "netsuite_transactions.csv":     "NetSuite SuiteAnalytics ODBC",
}

print(f"{'File':<40} {'Rows':>8}  {'Source'}")
print("-" * 75)
for fname, source in files.items():
    path = f"{OUTPUT}/{fname}"
    if os.path.exists(path):
        rows = sum(1 for _ in open(path)) - 1
        size = os.path.getsize(path) / 1024
        print(f"  {fname:<38} {rows:>8,}  {source}")
print("\n✅ All data ready for modeling.")